In [5]:
import pytz
from datetime import datetime
# Assuming you already have the scraped data in a DataFrame
# Convert Forex Factory time to MT5 time zone
def convert_time_to_mt5(df):
    # Define the time zones
    eastern_tz = pytz.timezone('America/New_York')  # Forex Factory
    mt5_tz = pytz.timezone('Etc/GMT-3')  # MT5 time zone

    # Convert the time column
    def convert_time(row):
        # Parse the time
        time_str = row['time']
        naive_time = datetime.strptime(time_str, "%I:%M%p")  # Convert to naive datetime
        
        # Localize to Eastern Time
        localized_time = eastern_tz.localize(naive_time)
        
        # Convert to MT5 time zone
        mt5_time = localized_time.astimezone(mt5_tz)
        
        return mt5_time.strftime("%I:%M %p")  # Return formatted string

    # Apply the conversion
    df['time'] = df.apply(convert_time, axis=1)

def news_fetch():
    try:
        from selenium import webdriver
        from selenium.webdriver.common.by import By
        driver = webdriver.Chrome()
    except:
        print ("AF: No Chrome webdriver installed")
        driver = webdriver.Chrome(ChromeDriverManager().install())

    import time
    import json
    import pandas as pd
    from datetime import datetime
    from config import ALLOWED_ELEMENT_TYPES,ICON_COLOR_MAP
    from utils import reformat_scraped_data
    from webdriver_manager.chrome import ChromeDriverManager

    driver.get("https://www.forexfactory.com/calendar")

    month =  datetime.now().strftime("%B")

    table = driver.find_element(By.CLASS_NAME, "calendar__table")

    data = []
    previous_row_count = 0
    # Scroll down to the end of the page
    while True:
        # Record the current scroll position
        before_scroll = driver.execute_script("return window.pageYOffset;")
        
        # Scroll down a fixed amount
        driver.execute_script("window.scrollTo(0, window.pageYOffset + 500);")
        
        # Wait for a short moment to allow content to load
        time.sleep(2)
        
        # Record the new scroll position
        after_scroll = driver.execute_script("return window.pageYOffset;")
        
        # If the scroll position hasn't changed, we've reached the end of the page
        if before_scroll == after_scroll:
            break

    # Now that we've scrolled to the end, collect the data
    for row in table.find_elements(By.TAG_NAME, "tr"):
        row_data = []
        for element in row.find_elements(By.TAG_NAME, "td"):
            class_name = element.get_attribute('class')
            if class_name in ALLOWED_ELEMENT_TYPES:
                if element.text:
                    row_data.append(element.text)
                elif "calendar__impact" in class_name:
                    impact_elements = element.find_elements(By.TAG_NAME, "span")
                    for impact in impact_elements:
                        impact_class = impact.get_attribute("class")
                        color = ICON_COLOR_MAP[impact_class]
                    if color:
                        row_data.append(color)
                    else:
                        row_data.append("impact")

        if len(row_data):
            data.append(row_data)

    reformat_scraped_data(data,month)
    ds = pd.read_csv(f'{month}_news.csv', parse_dates=["date"], index_col=0)
    ds = ds[ds['currency'] == 'USD']
    ds = ds[ds['impact'] == 'red']
    convert_time_to_mt5(ds)
    return ds

In [6]:
news_fetch()

/var/folders/21/7qb31d8x5zq3gyb3nyd5wyxh0000gn/T/ipykernel_8740/1063109480.py:92: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  ds = pd.read_csv(f'{month}_news.csv', parse_dates=["date"], index_col=0)


,time,currency,impact,event
date,,,,
Sep 23,05:41 PM,USD,red,Flash Manufacturing PMI
Sep 23,05:41 PM,USD,red,Flash Services PMI
Sep 24,05:56 PM,USD,red,CB Consumer Confidence
Sep 26,04:26 PM,USD,red,Final GDP q/q
Sep 26,04:26 PM,USD,red,Unemployment Claims
Sep 26,05:16 PM,USD,red,Fed Chair Powell Speaks
Sep 27,04:26 PM,USD,red,Core PCE Price Index m/m
